<a href="https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahmarani/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content page's daily performance record, for one client, on one report date. Time window: month = 2026-03 (a mid-panel month, avoiding the sealed final month of June 2026).

In [5]:
print("Unit of analysis: one row = one content page's daily performance record (client + content + report date)")
print("Time window: month 2026-03, a mid-panel month")

Unit of analysis: one row = one content page's daily performance record (client + content + report date)
Time window: month 2026-03, a mid-panel month


## 2. Fields: feature / label / context / excluded
1. Features: impressions, clicks, avg_position, ctr, sessions, engagement_rate (all known before the decision point)
2. Label/proxy: whether the page's trend later shows decline (to be defined precisely in modeling weeks)
3. Context: client_hash_id, content_hash_id (used for joins/grouping only, not as features)
4. Excluded: any product decision flags like health_score or priority_score — these encode FlyRank's own existing judgment and would leak the answer if used as a feature

In [6]:
print("Features: impressions, clicks, avg_position, ctr, sessions, engagement_rate (known before the decision point)")
print("Label/proxy: whether the page trend later shows decline")
print("Context only: client_hash_id, content_hash_id")
print("Excluded: product decision flags such as health_score or priority_score")

Features: impressions, clicks, avg_position, ctr, sessions, engagement_rate (known before the decision point)
Label/proxy: whether the page trend later shows decline
Context only: client_hash_id, content_hash_id
Excluded: product decision flags such as health_score or priority_score


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

base = "hf://datasets/FlyRank/internship-warehouse"

# Query 1: Grain check — one row per client+content+date?
q1 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as cnt
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1,2,3
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate grain rows found:", len(q1))

# Query 2: Row count and date span for this slice
q2 = con.sql(f"""
    SELECT COUNT(*) as row_count, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(q2)

# Query 3: Availability check with IS TRUE
q3 = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM read_parquet('{base}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(q3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 0
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  ga4_available_rows
0     9841378            413966.0


## 4. Data limits

This data has a few structural limits worth naming:

1. Unbalanced history — different clients have different amounts of tracking history (some 12+ months, some much less), so early rows for newer clients may look artificially "flat" simply because tracking just started, not because nothing was happening.
2. GSC-only early rows — before a client's GA4 start date, only search data (impressions, clicks, position) is available; engagement/session metrics are missing, not zero. Treating a missing value as zero would be wrong.
3. Window overlap risk — since this is a monthly snapshot (2026-03), any feature must be built only from information available within or before this window; nothing from later months should leak in when building features or labels.
4. No causal claims possible — this data shows what happened, not why. It can support "this page is declining" but never "refreshing this page will fix it" without a real experiment.

In [7]:
print("Limits: unbalanced client history; GSC-only early rows; window overlap risk; no causal claims")

Limits: unbalanced client history; GSC-only early rows; window overlap risk; no causal claims


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.